In [3]:
import numpy as np
import pandas as pd
import joblib
from lightgbm import LGBMClassifier
from sklearn.metrics import classification_report, confusion_matrix
import mlflow
import mlflow.lightgbm

X_train = np.load("../data/processed/X_train_multiclass.npy")
X_val = np.load("../data/processed/X_val_multiclass.npy")
y_train = np.load("../data/processed/y_train_multiclass.npy")
y_val = np.load("../data/processed/y_val_multiclass.npy")

label_encoder = joblib.load("../data/processed/label_encoder_multiclass.joblib")
class_names = label_encoder.classes_

print(X_train.shape, y_train.shape)
print(class_names)

(1764506, 78) (1764506,)
['BENIGN' 'Bot' 'DDoS' 'DoS GoldenEye' 'DoS Hulk' 'DoS Slowhttptest'
 'DoS slowloris' 'FTP-Patator' 'PortScan' 'SSH-Patator'
 'Web Attack - Brute Force' 'Web Attack - XSS']


In [4]:
lgbm_multi = LGBMClassifier(
    objective='multiclass',
    num_class=len(class_names),
    n_estimators=150,
    max_depth=10,
    class_weight='balanced',
    random_state=42,
    n_jobs=-1,
    verbose=-1
)

lgbm_multi.fit(X_train, y_train)
print("Multiclass LightGBM training complete.")

Multiclass LightGBM training complete.


In [5]:
y_val_pred = lgbm_multi.predict(X_val)

print(classification_report(y_val, y_val_pred, target_names=class_names))

                          precision    recall  f1-score   support

                  BENIGN       1.00      1.00      1.00    314258
                     Bot       0.73      0.89      0.80       292
                    DDoS       1.00      1.00      1.00     19202
           DoS GoldenEye       0.99      1.00      0.99      1543
                DoS Hulk       1.00      1.00      1.00     25927
        DoS Slowhttptest       0.99      0.99      0.99       784
           DoS slowloris       0.99      0.99      0.99       808
             FTP-Patator       1.00      1.00      1.00       890
                PortScan       0.99      1.00      0.99     13604
             SSH-Patator       1.00      1.00      1.00       483
Web Attack - Brute Force       0.77      0.77      0.77       220
        Web Attack - XSS       0.48      0.50      0.49        98

                accuracy                           1.00    378109
               macro avg       0.91      0.93      0.92    378109
        

In [6]:
cm = confusion_matrix(y_val, y_val_pred)
cm_df = pd.DataFrame(cm, index=class_names, columns=class_names)
print(cm_df)

                          BENIGN  Bot   DDoS  DoS GoldenEye  DoS Hulk  \
BENIGN                    313967   98      7              2        35   
Bot                           32  260      0              0         0   
DDoS                           1    0  19201              0         0   
DoS GoldenEye                  0    0      0           1539         3   
DoS Hulk                       4    0      0             10     25909   
DoS Slowhttptest               0    0      0              0         0   
DoS slowloris                  2    0      0              1         0   
FTP-Patator                    0    0      0              0         0   
PortScan                       2    0      0              0         6   
SSH-Patator                    0    0      0              0         0   
Web Attack - Brute Force       1    0      0              0         0   
Web Attack - XSS               0    0      0              0         0   

                          DoS Slowhttptest  DoS sl

In [7]:
mlflow.set_tracking_uri("http://127.0.0.1:5000")
mlflow.set_experiment("network-anomaly-multiclass")

with mlflow.start_run(run_name="lightgbm_multiclass"):
    mlflow.log_param("model_type", "LightGBM")
    mlflow.log_param("n_estimators", 150)
    mlflow.log_param("max_depth", 10)
    mlflow.log_param("num_classes", len(class_names))
    mlflow.lightgbm.log_model(lgbm_multi, "model")
    print("Logged to MLflow.")

2026/09/13 23:09:38 INFO mlflow.tracking.fluent: Experiment with name 'network-anomaly-multiclass' does not exist. Creating a new experiment.
2026/09/13 23:09:38 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


Logged to MLflow.
🏃 View run lightgbm_multiclass at: http://127.0.0.1:5000/#/experiments/2/runs/ac733fa88e9c42279502819521bf6ce0
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/2


In [8]:
joblib.dump(lgbm_multi, "../data/processed/lgbm_multiclass.joblib")
print("Multiclass model saved.")

Multiclass model saved.


In [1]:
import pandas as pd
import json
import gc

df = pd.read_parquet("../data/processed/cleaned_flows.parquet")

rare_classes = ['Heartbleed', 'Infiltration', 'Web Attack - Sql Injection']
df = df[~df['Label'].isin(rare_classes)]

feature_cols = [c for c in df.columns if c not in ['Label', 'Label_binary']]
with open("../data/processed/feature_columns.json", "w") as f:
    json.dump(feature_cols, f)

print(f"Saved {len(feature_cols)} feature column names.")

del df
gc.collect()

Saved 78 feature column names.


395

In [2]:
import numpy as np
import json

X_test = np.load("../data/processed/X_test_binary.npy")

with open("../data/processed/feature_columns.json") as f:
    feature_cols = json.load(f)

# grab one row, convert to plain list for JSON
sample_row = X_test[0].tolist()
print(json.dumps({"features": sample_row}))

{"features": [-0.4339635349826175, 0.1964203552731991, -0.0067303941001499714, -0.00815654072507052, -0.05068358637528562, -0.007623465106119505, -0.26432397121293655, -0.3167607310244144, -0.29275700953599904, -0.2135440354336418, -0.47815662263162406, -0.608394659087503, -0.5383803074786272, -0.427166160409999, -0.05279201166798931, -0.23324704851426378, 0.4087437760852682, 0.17368433729856156, -0.007356572810847224, -0.056311368502350106, 0.20640670608753478, 0.2961725611748342, 0.149903322446678, 0.0025963509816041355, -0.12506674047682415, 0.3104044804411574, 0.8713374969205941, -0.2517375320847442, 0.27237549833792013, 1.0383507535590903, -0.2259204430860346, 0.0, -0.005376254197939126, 0.0, 0.001457345458123585, 0.0017488569176876887, -0.21127914314548693, -0.1714668206998099, -0.6557533389086363, -0.4865649221803657, -0.5879164773662942, -0.4838002629476152, -0.31418949473895197, -0.18250034495606043, -0.2259204430860346, -0.016288044921378354, -0.6506740309979683, 1.4858732966

In [3]:
import numpy as np
import pandas as pd
import json

X_test = np.load("../data/processed/X_test_binary.npy")
y_test = pd.read_csv("../data/processed/y_test_binary.csv")['Label_binary']

# find index of an actual attack sample
attack_idx = y_test[y_test == 'ATTACK'].index[0]
sample_row = X_test[attack_idx].tolist()

print(json.dumps({"features": sample_row}))

{"features": [-0.453051366422891, -0.46991834667510146, -0.009158581117552628, -0.007243606190086317, -0.05112025100560164, -0.0029529677778166413, -0.27884014825924813, -0.3167607310244144, -0.28015669079268407, -0.22565427044235853, 5.2064383912504075, -0.608394659087503, 4.048966964994522, 6.144196679984086, -0.037727547900638154, -0.232058247241844, -0.30745991123617983, -0.3861853803951147, -0.3996380864494906, -0.05631534540292757, -0.4619694572133658, -0.29144489873851, -0.36160009241874974, -0.39339523447675373, -0.1250418428658033, -0.36666500861066714, -0.2150323024259054, -0.24933291450520736, -0.2890290874133955, -0.12334516114567634, -0.2259204430860346, 0.0, -0.005376254197939126, 0.0, 0.0014540362903707233, 0.0017464179957839027, -0.21074545191022523, -0.1678391126014978, -0.6557533389086363, 4.959150121151995, 3.968479718863909, 5.696741698486776, 9.348490806942438, -0.18250034495606043, -0.2259204430860346, -0.016288044921378354, 1.5368678514282403, -0.673004893651289,

In [5]:
raw_features = [float(v) for v in attack_row[feature_cols].tolist()]

print(json.dumps({"features": raw_features}))
print(f"\nActual label: {attack_row['Label']}")

{"features": [80.0, 1293792.0, 3.0, 7.0, 26.0, 11607.0, 20.0, 0.0, 8.666666984558105, 10.263202667236328, 5840.0, 0.0, 1658.142822265625, 2137.297119140625, 8991.398927, 7.72921768, 143754.6667, 430865.8067, 1292730.0, 2.0, 747.0, 373.5, 523.9661249, 744.0, 3.0, 1293746.0, 215624.3333, 527671.9348, 1292730.0, 2.0, 0.0, 0.0, 0.0, 0.0, 72.0, 152.0, 2.318765304, 5.410452376, 0.0, 5840.0, 1057.54541015625, 1853.4375, 3435230.673, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 2.0, 1163.300048828125, 8.666666984558105, 1658.142822265625, 72.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 3.0, 26.0, 7.0, 11607.0, 8192.0, 229.0, 2.0, 20.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]}

Actual label: DDoS
